# Gaussian Shading Benchmark — Google Colab
**Generation-based (Xiong et al. — truncated-Gaussian latents + ChaCha20).**  
Runs the **14 WAVES attacks** from `benchmark_attacks.py`, reports watermark recovery (bit accuracy), AUROC, and TPR@1% FPR.

### Setup
1. Runtime → **GPU** (A100 recommended; T4 works but slower)
2. Clones [Gaussian-Shading](https://github.com/bsmhmmlf/Gaussian-Shading) + [wavess](https://github.com/ademladhari/wavess)
3. Optional: results saved to `MyDrive/wavess_results/gaussian_shading/`

**Note:** Each image uses its own watermark key (per official protocol). Detection scores are per-image `eval_watermark` recovery rates.

In [ ]:
# ── 1. Clone repos + optional Drive mount ────────────────────────────────────
import os, re, sys, subprocess
from pathlib import Path

GS_REPO    = '/content/Gaussian-Shading'
GS_URL     = 'https://github.com/bsmhmmlf/Gaussian-Shading.git'
WAVES_ROOT = '/content/wavess'
WAVES_URL  = 'https://github.com/ademladhari/wavess.git'

if not os.path.isdir(GS_REPO):
    print('Cloning Gaussian-Shading…')
    subprocess.run(['git', 'clone', '--depth', '1', GS_URL, GS_REPO], check=True)
else:
    print('Gaussian-Shading already present')

if not os.path.isfile(f'{WAVES_ROOT}/benchmark_attacks.py'):
    print('Cloning wavess (for benchmark_attacks.py + prompts)…')
    subprocess.run(['git', 'clone', '--depth', '1', WAVES_URL, WAVES_ROOT], check=True)
else:
    print('wavess already present')

for p in [GS_REPO, WAVES_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(GS_REPO)

SAVE_TO_DRIVE = True
DRIVE_OUTPUT  = '/content/drive/MyDrive/wavess_results/gaussian_shading'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)

_ba = f'{WAVES_ROOT}/benchmark_attacks.py'
if not os.path.isfile(_ba):
    raise FileNotFoundError(
        f'Missing {_ba} — push benchmark_attacks.py to GitHub, then re-clone wavess.'
    )
print('Repos OK')

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
!pip install -q diffusers==0.38.0 transformers accelerate pycryptodome \
               scikit-image scikit-learn tqdm scipy datasets

In [ ]:
# ── 3. Patch Gaussian-Shading for modern Colab (fp32, no hard-coded .cuda()) ─
import re

def patch_file(path, subs):
    text = Path(path).read_text(encoding='utf-8')
    orig = text
    for pat, rep in subs:
        text = re.sub(pat, rep, text)
    if text != orig:
        Path(path).write_text(text, encoding='utf-8')
        print('patched', path)

def patch_clip_import(path):
    text = Path(path).read_text(encoding='utf-8')
    old = 'from transformers import CLIPFeatureExtractor, CLIPTextModel, CLIPTokenizer'
    new = (
        'try:\n'
        '    from transformers import CLIPFeatureExtractor, CLIPTextModel, CLIPTokenizer\n'
        'except ImportError:  # transformers >= 4.40\n'
        '    from transformers import CLIPImageProcessor as CLIPFeatureExtractor, CLIPTextModel, CLIPTokenizer'
    )
    if old in text and new not in text:
        Path(path).write_text(text.replace(old, new, 1), encoding='utf-8')
        print('patched CLIP import', path)

def patch_diffusers_safety_init(path):
    text = Path(path).read_text(encoding='utf-8')
    guard = (
        '        if isinstance(safety_checker, bool):\n'
        '            requires_safety_checker = safety_checker\n'
        '            safety_checker = None\n'
        '        if isinstance(feature_extractor, bool):\n'
        '            feature_extractor = None\n'
        '        if safety_checker is None:\n'
        '            requires_safety_checker = False\n\n'
    )
    if guard.strip() in text:
        print('already patched safety init', path)
        return
    needle = 'requires_safety_checker: bool = False,\n    ):\n\n        super('
    if needle in text:
        text = text.replace(needle, 'requires_safety_checker: bool = False,\n    ):\n' + guard + '        super(', 1)
        Path(path).write_text(text, encoding='utf-8')
        print('patched safety init', path)

fp16_subs = [
    (r'torch_dtype\s*=\s*torch\.float16,?', ''),
    (r'revision\s*=\s*["\']fp16["\'],?', ''),
]
for fname in ['inverse_stable_diffusion.py', 'modified_stable_diffusion.py', 'run_gaussian_shading.py']:
    p = Path(GS_REPO) / fname
    if p.is_file():
        patch_file(p, fp16_subs)
        if fname in ('inverse_stable_diffusion.py', 'modified_stable_diffusion.py'):
            patch_clip_import(p)
            patch_diffusers_safety_init(p)

wm_path = Path(GS_REPO) / 'watermark.py'
wm = wm_path.read_text(encoding='utf-8')
wm = wm.replace('.cuda()', '.to(device)')
wm = wm.replace('.half()', '.float()')
if 'device = ' not in wm:
    wm = wm.replace(
        'class Gaussian_Shading_chacha:',
        "device = 'cuda' if torch.cuda.is_available() else 'cpu'\n\nclass Gaussian_Shading_chacha:",
        1,
    )
wm_path.write_text(wm, encoding='utf-8')
print('watermark.py patched for device + float32')
print('Patching done.')

In [ ]:
# ── 4. Config — edit here ────────────────────────────────────────────────────
N_IMAGES       = 100
DETECT_BATCH   = 1          # DDIM inversion is serial; batch only groups tqdm
SEED           = 42
DEVICE         = 'cuda'
MODEL_ID       = 'sd2-community/stable-diffusion-2-1-base'
PROMPTS_FILE   = f'{WAVES_ROOT}/tree-ring/prompts.txt'
OUTPUT_DIR     = f'{WAVES_ROOT}/gaussian_shading/outputs_benchmark_colab'

# Gaussian Shading hyper-params (paper defaults)
CHANNEL_COPY   = 1
HW_COPY        = 8
FPR            = 1e-6
USER_NUMBER    = 1_000_000
USE_CHACHA     = True

GUIDANCE       = 7.5
NUM_STEPS      = 50
DETECT_STEPS   = 50
IMAGE_LEN      = 512

In [ ]:
# ── 5. Imports ────────────────────────────────────────────────────────────────
import csv, json
from time import time

import numpy as np
import torch
from diffusers import DPMSolverMultistepScheduler
from PIL import Image
from sklearn import metrics as sk_metrics
from tqdm.auto import tqdm

from benchmark_attacks import ATTACKS, apply_attack_rgb
from inverse_stable_diffusion import InversableStableDiffusionPipeline
from image_utils import transform_img, set_random_seed
from watermark import Gaussian_Shading_chacha, Gaussian_Shading

device = torch.device(DEVICE)
print('Imports OK | GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')

In [ ]:
# ── 6. Helpers ────────────────────────────────────────────────────────────────
def load_prompts(path, n):
    lines = [l.strip() for l in Path(path).read_text('utf-8').splitlines() if l.strip()]
    if len(lines) < n:
        lines = (lines * ((n // len(lines)) + 1))[:n]
    return lines[:n]

def make_watermark():
    if USE_CHACHA:
        return Gaussian_Shading_chacha(CHANNEL_COPY, HW_COPY, FPR, USER_NUMBER)
    return Gaussian_Shading(CHANNEL_COPY, HW_COPY, FPR, USER_NUMBER)

def auroc_tpr(pos, neg, fpr=0.01):
    y = np.concatenate([np.zeros(neg.size, np.int32), np.ones(pos.size, np.int32)])
    s = np.concatenate([neg, pos])
    auc = float(sk_metrics.roc_auc_score(y, s))
    fp, tp, _ = sk_metrics.roc_curve(y, s, pos_label=1)
    idx = np.where(fp < fpr)[0]
    return auc, float(tp[idx[-1]]) if idx.size else float(tp[0])

print('Helpers defined.')

In [ ]:
# ── 7. Load SD pipeline ───────────────────────────────────────────────────────
scheduler = DPMSolverMultistepScheduler.from_pretrained(MODEL_ID, subfolder='scheduler')
pipe = InversableStableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    scheduler=scheduler,
    torch_dtype=torch.float32,
    safety_checker=None,
    feature_extractor=None,
).to(DEVICE)
pipe.set_progress_bar_config(disable=True)

tester_prompt = ''
text_emb = pipe.get_text_embedding(tester_prompt)
print('Pipeline loaded:', MODEL_ID)

In [ ]:
# ── 8. Generate watermarked + clean pairs (per-image progress) ────────────────
prompts = load_prompts(PROMPTS_FILE, N_IMAGES)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

records = []
print(f'Generating {N_IMAGES} pairs (watermarked + clean)…')
t0 = time()

for i in tqdm(range(N_IMAGES), desc='generate', unit='img'):
    gen_seed = SEED + i
    prompt   = prompts[i]

    # watermarked
    wm = make_watermark()
    set_random_seed(gen_seed)
    init_latents_w = wm.create_watermark_and_return_w().to(dtype=pipe.unet.dtype, device=device)
    wm_out = pipe(
        prompt,
        num_images_per_prompt=1,
        guidance_scale=GUIDANCE,
        num_inference_steps=NUM_STEPS,
        height=IMAGE_LEN,
        width=IMAGE_LEN,
        latents=init_latents_w,
    )

    # clean (same prompt, random latents — no GS watermark)
    set_random_seed(gen_seed + 10_000)
    gen = torch.Generator(device=device).manual_seed(gen_seed + 10_000)
    clean_latents = pipe.get_random_latents(height=IMAGE_LEN, width=IMAGE_LEN, generator=gen)
    clean_out = pipe(
        prompt,
        num_images_per_prompt=1,
        guidance_scale=GUIDANCE,
        num_inference_steps=NUM_STEPS,
        height=IMAGE_LEN,
        width=IMAGE_LEN,
        latents=clean_latents,
    )

    records.append({
        'prompt': prompt,
        'wm': wm,
        'wm_pil': wm_out.images[0].convert('RGB'),
        'clean_pil': clean_out.images[0].convert('RGB'),
        'seed': gen_seed,
    })

    if (i + 1) % 10 == 0:
        elapsed = time() - t0
        rate = (i + 1) / elapsed
        eta_h = (N_IMAGES - i - 1) / rate / 3600
        print(f'  [{i+1}/{N_IMAGES}] {rate:.3f} img/s — ETA {eta_h:.1f} h', flush=True)

print(f'Done: {len(records)} pairs in {(time()-t0)/3600:.2f} h')

In [ ]:
# ── 9. Detection helper (DDIM invert + eval_watermark) ───────────────────────
@torch.no_grad()
def detect_score(wm_obj, pil_img):
    """Return watermark bit-recovery rate in [0, 1] (higher = more watermarked)."""
    im_t = transform_img(pil_img, target_size=IMAGE_LEN).unsqueeze(0).to(text_emb.dtype).to(device)
    lat_img = pipe.get_image_latents(im_t, sample=False)
    rev = pipe.forward_diffusion(
        latents=lat_img,
        text_embeddings=text_emb,
        guidance_scale=1.0,
        num_inference_steps=DETECT_STEPS,
    )
    return float(wm_obj.eval_watermark(rev))

def detect_batch(records_slice, pil_key):
    return [detect_score(r['wm'], r[pil_key]) for r in records_slice]

sanity = detect_score(records[0]['wm'], records[0]['wm_pil'])
print(f'Sanity (image 0, no attack): recovery = {sanity:.4f}')

neg_scores = []
print('Scoring negatives (clean images)…')
for i in tqdm(range(0, len(records), DETECT_BATCH), desc='negatives'):
    neg_scores.extend(detect_batch(records[i:i+DETECT_BATCH], 'clean_pil'))
neg_arr = np.asarray(neg_scores, dtype=np.float64)
print(f'Neg recovery mean: {neg_arr.mean():.4f}')

In [ ]:
# ── 10. Per-attack evaluation (14 WAVES attacks) ─────────────────────────────
rows = []

for spec in ATTACKS:
    pos_scores = []
    attacked = [apply_attack_rgb(spec.name, r['wm_pil'], seed=i) for i, r in enumerate(records)]

    for i in range(0, len(records), DETECT_BATCH):
        batch_recs = records[i:i+DETECT_BATCH]
        batch_atk  = attacked[i:i+DETECT_BATCH]
        for rec, atk in zip(batch_recs, batch_atk):
            pos_scores.append(detect_score(rec['wm'], atk))

    pos_arr = np.asarray(pos_scores, dtype=np.float64)
    auc, tpr1 = auroc_tpr(pos_arr, neg_arr)
    recovery = float(pos_arr.mean())
    ber = 100.0 * (1.0 - recovery)   # watermark-pattern error rate

    row = {
        'method': 'gaussian_shading',
        'attack': spec.name,
        'description': spec.description,
        'n_images': len(records),
        'bit_accuracy': recovery,
        'BER': ber / 100.0,
        'AUROC': auc,
        'TPR_at_1pct_FPR': tpr1,
    }
    rows.append(row)
    print(f"  {spec.name:22s}: recovery={recovery:.3f}  BER={ber:5.1f}%  AUROC={auc:.3f}  TPR@1%={tpr1:.3f}")

print('\nAll attacks done.')

In [ ]:
# ── 11. Save results ──────────────────────────────────────────────────────────
csv_path  = Path(OUTPUT_DIR) / 'results.csv'
json_path = Path(OUTPUT_DIR) / 'results.json'

with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump({
        'method': 'gaussian_shading',
        'n_images': len(records),
        'model_id': MODEL_ID,
        'chacha': USE_CHACHA,
        'channel_copy': CHANNEL_COPY,
        'hw_copy': HW_COPY,
        'fpr': FPR,
        'sanity_recovery': sanity,
        'attacks': rows,
    }, f, indent=2)

print(f'Saved to {OUTPUT_DIR}')
if SAVE_TO_DRIVE:
    import shutil
    shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT, dirs_exist_ok=True)
    print(f'Copied to Drive: {DRIVE_OUTPUT}')

import pandas as pd
df = pd.DataFrame(rows)[['attack', 'bit_accuracy', 'BER', 'AUROC', 'TPR_at_1pct_FPR']]
df['BER %'] = (df['BER'] * 100).round(1)
df = df.drop(columns=['BER'])
df.columns = ['Attack', 'Recovery', 'BER %', 'AUROC', 'TPR@1%FPR']
print(df.to_string(index=False))